# 🎨 Graphic Design & Marketing Internship Finder
**Searches in-person jobs within 50 miles of ZIP 45044 (Mason, OH) + remote jobs nationwide**
**Sales jobs & spam companies (5+ listings) are automatically filtered out.**
**Jobs are synced to Google Sheets — edits on the live website persist here too.**

### Setup Instructions
1. Go to [developer.adzuna.com](https://developer.adzuna.com/) and register for a free account
2. Copy your **App ID** and **App Key** from the dashboard
3. Paste your **Apps Script URL** and **Adzuna credentials** into **Cell 2** below
4. Run all cells (Runtime → Run all)

---

In [12]:
# ── Cell 1: Install dependencies ────────────────────────────────────────────────────
!pip install requests pandas openpyxl --quiet
print("✅ Dependencies ready")

✅ Dependencies ready


In [13]:
# ── Cell 2: Load credentials from Colab Secrets ──────────────────────────────
# Secrets are stored in the 🔑 sidebar — never hardcoded, never on GitHub.
# First time setup: add ADZUNA_APP_ID, ADZUNA_APP_KEY, APPS_SCRIPT_URL
# in the Colab Secrets panel and toggle "Notebook access" on for each.

from google.colab import userdata

try:
    APP_ID          = userdata.get('ADZUNA_APP_ID')
    APP_KEY         = userdata.get('ADZUNA_APP_KEY')
    APPS_SCRIPT_URL = userdata.get('APPS_SCRIPT_URL')
    print("✅ Credentials loaded from Secrets")
except Exception as e:
    raise RuntimeError(
        f"❌ Could not load credentials: {e}\n"
        "Fix: click the 🔑 icon in the left sidebar → add ADZUNA_APP_ID, "
        "ADZUNA_APP_KEY, and APPS_SCRIPT_URL → toggle 'Notebook access' ON for each."
    )

# ── Location config ───────────────────────────────────────────────────────────
ZIP_CODE     = "45044"
RADIUS_MILES = 50
MAX_RESULTS  = 50

# ── Toggle remote search ──────────────────────────────────────────────────────
INCLUDE_REMOTE = True

# ── Sales job exclusion ───────────────────────────────────────────────────────
SALES_EXCLUDE_KEYWORDS = [
    "sales", "account executive", "business development", "bdr", "sdr",
    "revenue", "quota", "cold call", "door-to-door", "commission",
    "insurance agent", "financial advisor", "realtor", "mortgage",
    "loan officer", "wealth management","accounting","finance"
]

# ── Search queries ────────────────────────────────────────────────────────────
SEARCH_QUERIES = [
    # ── Graphic Design ────────────────────────────────────────────────────────
    "graphic design intern",
    "graphic designer intern",
    "visual design intern",
    "marketing design intern",
    "creative design intern",
    "brand design intern",
    "print design intern",
    "illustration intern",
    "motion graphics intern",
    "UI design intern",
    "UX design intern",
    "product design intern",
    "web design intern",
    "art direction intern",
    "packaging design intern",
    "publication design intern",
    "digital content design intern",
    "AR design",
    "VR design",
    "3D design",

    # ── Marketing (General) ───────────────────────────────────────────────────
    "marketing intern",
    "marketing communications intern",
    "brand marketing intern",
    "content marketing intern",
    "digital marketing intern",
    "SEM intern",
    "paid media intern",
    "media planning intern",
    "PR intern",
    "public relations intern",
    "communications intern",
    "copywriting intern",
    "content creation intern",
    "content strategy intern",
    "creative writing intern",
    # ── Social Media ──────────────────────────────────────────────────────────
    "social media intern",
    "social media design intern",
    "social media marketing intern",
    "social media coordinator intern",
    "Instagram marketing intern",
    "TikTok marketing intern",
    "influencer marketing intern",
    "community management intern",
    # ── Video / Photo / Multimedia ────────────────────────────────────────────
    "video production intern",
    "video editing intern",
    "videography intern",
    "photography intern",
    "multimedia design intern",
    "creative media intern",
    "podcast production intern",
    # ── Creative / Agency ─────────────────────────────────────────────────────
    "creative agency intern",
    "advertising intern",
    "creative strategist intern",
    "brand strategy intern",
    "creative services intern",
    "integrated marketing intern",
    "event marketing intern",
]

print(f"✅ Configuration set — {len(SEARCH_QUERIES)} queries × "
      f"{'2 modes (local + remote)' if INCLUDE_REMOTE else '1 mode (local only)'}")

✅ Credentials loaded from Secrets
✅ Configuration set — 57 queries × 2 modes (local + remote)


In [14]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────────────────
import requests
import pandas as pd
import time
import json
import os
from IPython.display import display, HTML
from datetime import datetime

print("✅ Imports complete")

✅ Imports complete


In [15]:
# ── Cell 4: Search & filter functions ───────────────────────────────────────────────
def search_internships(query, location=None, radius_miles=50, max_results=50, page=1, remote=False):
    """
    Query the Adzuna Jobs API for internship listings.

    Args:
        query        : keyword string, e.g. 'graphic design intern'
        location     : ZIP code or city name string (ignored when remote=True)
        radius_miles : search radius (converted to km for the API)
        max_results  : results to return per page (max 50)
        page         : pagination page number (start at 1)
        remote       : if True, search for remote jobs nationwide

    Returns:
        (list of raw job result dicts, total count int)
    """
    base_url = f"https://api.adzuna.com/v1/api/jobs/us/search/{page}"

    params = {
        "app_id"           : APP_ID,
        "app_key"          : APP_KEY,
        "results_per_page" : max_results,
        "what"             : query,
        "sort_by"          : "date",
        "content-type"     : "application/json"
    }

    if remote:
        # Adzuna remote flag — no location/radius needed
        params["what"] = query + " remote"
    else:
        params["where"]    = location
        params["distance"] = int(radius_miles * 1.609)  # API uses km

    response = requests.get(base_url, params=params, timeout=10)

    # Surface a helpful error if credentials are wrong
    if response.status_code == 401:
        raise ValueError("❌ Authentication failed — double-check your APP_ID and APP_KEY in Cell 2.")

    response.raise_for_status()
    data = response.json()
    return data.get("results", []), data.get("count", 0)


def is_sales_job(job):
    """
    Return True if the job looks like a sales role based on title or description.
    Checks against SALES_EXCLUDE_KEYWORDS (case-insensitive).
    """
    haystack = " ".join([
        job.get("title", ""),
        job.get("description", ""),
    ]).lower()
    return any(kw.lower() in haystack for kw in SALES_EXCLUDE_KEYWORDS)


def parse_results(results, source_tag=""):
    """
    Convert raw Adzuna result dicts into a flat list of row dicts.
    source_tag: short label added to Location column, e.g. '🌐 Remote'
    """
    rows = []
    for job in results:
        # Salary: use min if available, otherwise blank
        sal_min = job.get("salary_min")
        sal_max = job.get("salary_max")
        if sal_min and sal_max:
            salary = f"${sal_min:,.0f} – ${sal_max:,.0f}"
        elif sal_min:
            salary = f"${sal_min:,.0f}+"
        else:
            salary = "Not listed"

        # Date formatting
        raw_date = job.get("created", "")
        try:
            posted = datetime.fromisoformat(raw_date[:10]).strftime("%b %d, %Y")
        except Exception:
            posted = raw_date[:10] if raw_date else "N/A"

        location_str = job.get("location", {}).get("display_name", "N/A")
        if source_tag:
            location_str = f"{source_tag} | {location_str}"

        rows.append({
            "Title"      : job.get("title", "N/A"),
            "Company"    : job.get("company", {}).get("display_name", "N/A"),
            "Location"   : location_str,
            "Posted"     : posted,
            "Salary"     : salary,
            "URL"        : job.get("redirect_url", ""),
            "Description": job.get("description", "")[:300] + "..."  # truncate for preview
        })
    return rows


print("✅ Functions defined")

✅ Functions defined


In [16]:
# ── Cell 5: Run all searches (local + remote) ────────────────────────────────────────
all_raw_results = []   # raw dicts kept for full-description export
all_parsed_rows = []   # parsed row dicts (with source tag already applied)

# ── Pass 1: In-person jobs near ZIP_CODE ─────────────────────────────────────────────
print(f"📍 IN-PERSON SEARCH — within {RADIUS_MILES} miles of {ZIP_CODE}")
print("-" * 60)
for query in SEARCH_QUERIES:
    print(f"  🔍 '{query}'...", end=" ")
    try:
        results, total = search_internships(
            query=query,
            location=ZIP_CODE,
            radius_miles=RADIUS_MILES,
            max_results=MAX_RESULTS,
            remote=False
        )
        # Pre-filter sales jobs on raw results before storing
        clean = [r for r in results if not is_sales_job(r)]
        all_raw_results.extend(clean)
        all_parsed_rows.extend(parse_results(clean, source_tag="📍 Local"))
        print(f"{len(clean)} kept ({len(results)-len(clean)} sales filtered) of {total} total")
    except Exception as e:
        print(f"\n  ⚠️  Error: {e}")
    time.sleep(0.5)

# ── Pass 2: Remote jobs nationwide ───────────────────────────────────────────────────
if INCLUDE_REMOTE:
    print()
    print("🌐 REMOTE SEARCH — nationwide")
    print("-" * 60)
    for query in SEARCH_QUERIES:
        print(f"  🔍 '{query} remote'...", end=" ")
        try:
            results, total = search_internships(
                query=query,
                max_results=MAX_RESULTS,
                remote=True
            )
            clean = [r for r in results if not is_sales_job(r)]
            all_raw_results.extend(clean)
            all_parsed_rows.extend(parse_results(clean, source_tag="🌐 Remote"))
            print(f"{len(clean)} kept ({len(results)-len(clean)} sales filtered) of {total} total")
        except Exception as e:
            print(f"\n  ⚠️  Error: {e}")
        time.sleep(0.5)

#print()
#print(f"📦 Total raw results collected (pre-dedup): {len(all_raw_results)}")

📍 IN-PERSON SEARCH — within 50 miles of 45044
------------------------------------------------------------
  🔍 'graphic design intern'... 2 kept (0 sales filtered) of 2 total
  🔍 'graphic designer intern'... 2 kept (0 sales filtered) of 2 total
  🔍 'visual design intern'... 0 kept (0 sales filtered) of 0 total
  🔍 'marketing design intern'... 1 kept (0 sales filtered) of 1 total
  🔍 'creative design intern'... 1 kept (0 sales filtered) of 1 total
  🔍 'brand design intern'... 0 kept (0 sales filtered) of 0 total
  🔍 'print design intern'... 0 kept (0 sales filtered) of 0 total
  🔍 'illustration intern'... 0 kept (0 sales filtered) of 0 total
  🔍 'motion graphics intern'... 0 kept (0 sales filtered) of 0 total
  🔍 'UI design intern'... 0 kept (0 sales filtered) of 0 total
  🔍 'UX design intern'... 0 kept (0 sales filtered) of 0 total
  🔍 'product design intern'... 0 kept (0 sales filtered) of 0 total
  🔍 'web design intern'... 0 kept (0 sales filtered) of 0 total
  🔍 'art direction inter

In [17]:
# ── Cell 6: Parse, deduplicate, filter spam companies, and sort ──────────────────────
import pandas as pd

df = pd.DataFrame(all_parsed_rows)

# Deduplicate on URL (same job may appear across multiple queries)
df = df.drop_duplicates(subset="URL").reset_index(drop=True)

# ── Filter out companies with 5+ listings ─────────────────────────────────────────────
# Companies posting this many listings are usually staffing farms / aggregators,
# not real employers with a single open internship.
company_counts = df["Company"].value_counts()
spam_companies = company_counts[company_counts >= 5].index.tolist()
if spam_companies:
    print(f"🚫 Removing {len(spam_companies)} high-volume company/companies (5+ listings):")
    for c in spam_companies:
        print(f"   • {c} ({company_counts[c]} listings)")
    df = df[~df["Company"].isin(spam_companies)].reset_index(drop=True)

# Sort newest first
df = df.sort_values("Posted", ascending=False).reset_index(drop=True)

local_count  = df["Location"].str.startswith("📍").sum()
remote_count = df["Location"].str.startswith("🌐").sum()

print(f"\n✅ Unique listings after dedup + spam filter : {len(df)}")
print(f"   📍 In-person                              : {local_count}")
print(f"   🌐 Remote                                 : {remote_count}")
print()
df[["Title", "Company", "Location", "Posted", "Salary"]].head(10)

🚫 Removing 5 high-volume company/companies (5+ listings):
   • EBSCO Information Services (36 listings)
   • Olsson (7 listings)
   • CDM Smith (6 listings)
   • Gpac (5 listings)
   • Applied Research Solutions (5 listings)

✅ Unique listings after dedup + spam filter : 103
   📍 In-person                              : 74
   🌐 Remote                                 : 29



,Title,Company,Location,Posted,Salary
0,Assurance Manager - Cincinnati,GBQ Holdings,"📍 Local | Cincinnati, Hamilton County","Sep 13, 2025","$100,273 – $100,273"
1,Speech Language Pathologist,Sidney City SD,"📍 Local | Sidney, Shelby County","Nov 10, 2025","$58,861 – $58,861"
2,Fall 2026 Marketing Intern,Gmi Companies Inc,"📍 Local | Lebanon, Warren County","May 29, 2025","$19,774 – $19,774"
3,Engineering Intern,AeroVironment,"📍 Local | Wpafb, Greene County","Mar 28, 2026","$55,298 – $55,298"
4,Design Engineer - Equipment (Piqua),Tri-S Recruiters,"📍 Local | Piqua, Miami County","Mar 28, 2026","$105,559 – $105,559"
5,Municipal Project Manager (Columbus),OHM Advisors,"📍 Local | New Rome, Franklin County","Mar 27, 2026","$117,828 – $117,828"
6,Roadway Engineer (Cincinnati),Woolpert,"📍 Local | Norwood, Hamilton County","Mar 27, 2026","$76,703 – $76,703"
7,Roadway Engineer (Columbus),Woolpert,"📍 Local | New Rome, Franklin County","Mar 27, 2026","$100,346 – $100,346"
8,Engineering Intern,AeroVironment,"📍 Local | Wpafb, Greene County","Mar 27, 2026","$52,080 – $52,080"
9,Senior Engineer - Mechanical,Qualus,"📍 Local | Pisgah, Butler County","Mar 27, 2026","$82,941 – $82,941"


In [18]:
# ── Cell 7: Sync results to Google Sheets via Apps Script ───────────────────────────
#
# HOW PERSISTENCE WORKS
# ─────────────────────
# This cell POSTs all freshly-found jobs to your Apps Script web app, which
# writes them into the Google Sheet.  The Sheet is the single source of truth:
#   • New jobs are inserted with Status = 'Not Applied' and blank date/notes fields
#   • Existing jobs (matched by URL) only have their read-only columns refreshed
#   • Status / dates / notes you've edited on the website are NEVER overwritten
#
# The website reads from the same Sheet in real time.

STATUS_OPTIONS = ['Not Applied', 'Applied', 'Interviewing', 'Offer', 'Rejected', 'Withdrawn']

def sync_to_sheets(dataframe, apps_script_url):
    """
    POST all rows in `dataframe` to the Apps Script upsert endpoint.
    Returns the response dict {ok, inserted, updated}.
    """
    rows = []
    for _, row in dataframe.iterrows():
        rows.append({
            'url'        : row.get('URL', ''),
            'title'      : row.get('Title', ''),
            'company'    : row.get('Company', ''),
            'location'   : row.get('Location', ''),
            'posted'     : row.get('Posted', ''),
            'salary'     : row.get('Salary', ''),
            'description': row.get('Description', ''),
        })

    payload = json.dumps({'action': 'upsert', 'rows': rows})
    resp = requests.post(
        apps_script_url,
        data=payload,
        headers={'Content-Type': 'application/json'},
        timeout=60,          # Sheets writes can be slow on first call
    )
    resp.raise_for_status()
    return resp.json()


print(f"📤 Syncing {len(df)} jobs to Google Sheets...")
try:
    result = sync_to_sheets(df, APPS_SCRIPT_URL)
    if result.get('ok'):
        print(f"✅ Sync complete — {result['inserted']} new, {result['updated']} refreshed")
        print(f"🌐 Open your GitHub Pages site to see the updated tracker.")
    else:
        print(f"⚠️  Apps Script returned an error: {result.get('error')}")
except Exception as e:
    print(f"❌ Sync failed: {e}")
    print("   Double-check that APPS_SCRIPT_URL is set correctly in Cell 2.")

📤 Syncing 103 jobs to Google Sheets...
✅ Sync complete — 58 new, 45 refreshed
🌐 Open your GitHub Pages site to see the updated tracker.


In [19]:
# ── Cell 8: Quick filter preview (read-only, Sheet is source of truth) ───────────────
# This cell filters the current run's dataframe for a quick local preview.
# To see status/date edits, open the live website instead.
#
# Work-type: use '📍' for local only, '🌐' for remote only

FILTER_KEYWORD = "graphic"  # try: 'marketing', 'social', 'video', '🌐', '📍'

mask = (
    df["Title"].str.contains(FILTER_KEYWORD, case=False, na=False)
    | df["Company"].str.contains(FILTER_KEYWORD, case=False, na=False)
    | df["Location"].str.contains(FILTER_KEYWORD, case=False, na=False)
)
filtered_df = df[mask].reset_index(drop=True)
print(f"🔎 '{FILTER_KEYWORD}' → {len(filtered_df)} of {len(df)} jobs")
filtered_df[["Title", "Company", "Location", "Posted", "Salary"]]

🔎 'graphic' → 5 of 103 jobs


,Title,Company,Location,Posted,Salary
0,Graphic Design Intern (Summer 2026),GigFinesse,"🌐 Remote | Austin, Travis County","Mar 24, 2026","$45,223 – $45,223"
1,Graphic Designer II,Nelson,"📍 Local | Cincinnati, Hamilton County","Mar 21, 2026","$71,655 – $71,655"
2,Retail Graphic & Environmental Designer - Fair...,Cip International,"📍 Local | Lindenwald, Butler County","Mar 06, 2026","$64,979 – $64,979"
3,Motion Graphics Intern (IN PERSON OR REMOTE),Watertown Shamrocks,"🌐 Remote | Watertown, Codington County","Feb 28, 2026","$48,117 – $48,117"
4,Graphic Specialist/(2D/3D) Designer – Intermed...,SPS External,"🌐 Remote | Redstone Arsenal, Madison County","Feb 21, 2026","$75,938 – $75,938"


In [20]:
# ── Cell 9: Quick stats summary ───────────────────────────────────────────────────────
print("=" * 55)
print("📊 SEARCH SUMMARY")
print("=" * 55)
print(f"  ZIP code searched : {ZIP_CODE}")
print(f"  Radius            : {RADIUS_MILES} miles")
print(f"  Remote included   : {INCLUDE_REMOTE}")
print(f"  Queries run       : {len(SEARCH_QUERIES)} × {'2 modes' if INCLUDE_REMOTE else '1 mode'}")
print(f"  Unique listings   : {len(df)} (this run)")
print(f"    📍 In-person    : {df['Location'].str.startswith('📍').sum()}")
print(f"    🌐 Remote       : {df['Location'].str.startswith('🌐').sum()}")
print()

# Top companies
print("🏢 Top companies hiring (this run):")
print(df["Company"].value_counts().head(10).to_string())
print()

# Listings with salary info
with_salary = df[df["Salary"] != "Not listed"]
print(f"💰 Listings with salary info: {len(with_salary)} of {len(df)}")
if not with_salary.empty:
    print(with_salary[["Title", "Company", "Salary"]].to_string(index=False))

📊 SEARCH SUMMARY
  ZIP code searched : 45044
  Radius            : 50 miles
  Remote included   : True
  Queries run       : 57 × 2 modes
  Unique listings   : 103 (this run)
    📍 In-person    : 74
    🌐 Remote       : 29

🏢 Top companies hiring (this run):
Company
GBQ Holdings                   3
Amazon                         3
Childhood Cancer Society       3
Brilliant                      3
Magnum Piering                 2
AeroVironment                  2
Kroger                         2
University of Cincinnati       2
Crown Equipment Corporation    2
Walmart                        2

💰 Listings with salary info: 103 of 103
                                                                                          Title                                     Company              Salary
                                                                 Assurance Manager - Cincinnati                                GBQ Holdings $100,273 – $100,273
                                          

In [21]:
# ── Cell 10: Export tracker to Excel (pulls live data from Google Sheets) ────────────
# Fetches ALL tracked rows from the Sheet (including status/dates you've edited
# on the website) and downloads them as a timestamped .xlsx file.
from google.colab import files

print("📥 Fetching latest data from Google Sheets...")
try:
    # Apps Script redirects once — a Session follows it automatically.
    # We also set a generous timeout because Sheets can be slow on cold start.
    session = requests.Session()
    resp = session.get(APPS_SCRIPT_URL, allow_redirects=True, timeout=120)
    resp.raise_for_status()

    # Guard against empty body (redirect not followed properly)
    if not resp.text.strip():
        raise ValueError(
            "Apps Script returned an empty response.\n"
            "Fix: open Apps Script → Deploy → Manage deployments → "
            "click the pencil ✏ → change 'Who has access' to Anyone → "
            "click Deploy and copy the NEW url into Cell 2."
        )

    data = resp.json()
    if not data.get('ok'):
        raise ValueError(data.get('error', 'Unknown error from Apps Script'))

    sheet_rows = data['rows']
    export_df = pd.DataFrame(sheet_rows)

    # Friendly column names for Excel
    export_df = export_df.rename(columns={
        'url'          : 'URL',
        'title'        : 'Title',
        'company'      : 'Company',
        'location'     : 'Location',
        'posted'       : 'Posted',
        'salary'       : 'Salary',
        'status'       : 'Status',
        'date_applied' : 'Date Applied',
        'date_followup': 'Followed Up',
        'notes'        : 'Notes',
        'description'  : 'Description',
    })

    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    filename  = f'internships_{ZIP_CODE}_{timestamp}.xlsx'
    export_df.to_excel(filename, index=False)
    files.download(filename)
    print(f"✅ Downloaded: {filename} ({len(export_df)} rows)")

except Exception as e:
    print(f"❌ Export failed: {e}")

📥 Fetching latest data from Google Sheets...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded: internships_45044_20260328_1256.xlsx (153 rows)


---
## 🔧 Customization Tips

| What to change | Where | How |
|---|---|---|
| Adzuna credentials | Cell 2 | Replace `APP_ID` / `APP_KEY` |
| Apps Script URL | Cell 2 | Replace `APPS_SCRIPT_URL` |
| ZIP code / city | Cell 2 | Change `ZIP_CODE` |
| Search radius | Cell 2 | Change `RADIUS_MILES` |
| Toggle remote search | Cell 2 | Set `INCLUDE_REMOTE = False` |
| Add/remove job queries | Cell 2 | Edit `SEARCH_QUERIES` list |
| Tweak sales filter | Cell 2 | Edit `SALES_EXCLUDE_KEYWORDS` |
| Spam company threshold | Cell 6 | Change `>= 5` to a different number |
| Filter this run's results | Cell 8 | Change `FILTER_KEYWORD` |

### Architecture
```
Colab notebook
     │  POST /upsert
     ▼
Google Apps Script  ←──────────────────────────────┐
     │  read/write                                  │
     ▼                                     POST /update_field
Google Sheets (single source of truth)             │
     │  GET /rows                                   │
     ▼                                              │
GitHub Pages (index.html)  ────────────────────────┘
   Tabulator.js filterable table
   Status / dates / notes editable in browser
```

### Adding LLM analysis later
When you're ready, add a new Apps Script function:
```javascript
function analyzeJob(url) {
  // call Anthropic API, write score/summary to a new Sheet column
  // then add that column to index.html's Tabulator config
}
```
No changes needed to the Colab notebook or the Tabulator column definitions —
new columns in the Sheet automatically appear in the GET response.

### Adzuna Free Tier Limits
- ~250 API calls/month on the free plan
- 55 queries × 2 modes = **110 calls per run** (~2 runs/month)
- Set `INCLUDE_REMOTE = False` for 55 calls/run (~4 runs/month)

### Want More Results Per Query?
```python
for page_num in range(1, 4):  # pages 1, 2, 3
    results, _ = search_internships(query=query, location=ZIP_CODE, page=page_num)
    if not results:
        break
    all_raw_results.extend(results)
```